# Lesson 09: Simple Auth, RBAC Middleware, MCP Hosts & Container Deployment

In this notebook, you will:
1. Build a `FastAPI` + `PyJWT` RBAC middleware enforcing `viewer`, `developer`, and `admin` roles.
2. Test it with `fastapi.testclient.TestClient`.
3. Generate a hardened, non-root `Dockerfile` and `.vscode/mcp.json`.

In [ ]:
!pip install -q -U fastapi PyJWT httpx

In [ ]:
import jwt
import time
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient

SECRET = "colab-demo-rbac-secret-32-bytes"
app = FastAPI()

@app.middleware("http")
async def rbac_middleware(request: Request, call_next):
    if request.url.path == "/health":
        return await call_next(request)
    auth = request.headers.get("Authorization", "")
    if not auth.startswith("Bearer "):
        return JSONResponse(status_code=401, content={"error": "Missing Bearer token"})
    try:
        claims = jwt.decode(auth.split(" ", 1)[1], SECRET, algorithms=["HS256"])
        request.state.role = claims.get("role", "viewer")
    except jwt.PyJWTError:
        return JSONResponse(status_code=401, content={"error": "Invalid token"})
    return await call_next(request)

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/mcp/delete")
def delete_item(request: Request):
    if request.state.role != "admin":
        return JSONResponse(status_code=403, content={"error": "Admin role required"})
    return {"deleted": True}

client = TestClient(app)
viewer_tok = jwt.encode({"role": "viewer", "exp": int(time.time()) + 300}, SECRET, algorithm="HS256")
admin_tok = jwt.encode({"role": "admin", "exp": int(time.time()) + 300}, SECRET, algorithm="HS256")

print("No auth:", client.post("/mcp/delete").status_code)
print("Viewer:", client.post("/mcp/delete", headers={"Authorization": f"Bearer {viewer_tok}"}).status_code)
print("Admin:", client.post("/mcp/delete", headers={"Authorization": f"Bearer {admin_tok}"}).status_code)

### Challenge Exercise
Write a `%%writefile Dockerfile` cell using a two-stage `python:3.11-slim` build with a non-root `appuser` and a `HEALTHCHECK` hitting `/health`.